# MACE-MDP Raman Tutorial

This notebook presents an end-to-end **MACE-MDP** Raman workflow for a single molecule using the first structure in the example XYZ dataset.


## What Raman measures and what we compute

Raman scattering probes how polarizability ($\boldsymbol\alpha$) changes during a vibration. In this tutorial, these derivatives are evaluated with **MACE-MDP** models.

Equations used:

- Normal modes/frequencies from the Hessian (same route as IR).
- For each mode $k$, with derivative tensor $\partial\boldsymbol\alpha/\partial Q_k$:
  $$\bar\alpha'_k = \frac{1}{3}\,\mathrm{Tr}\!\left(\frac{\partial\boldsymbol\alpha}{\partial Q_k}\right)$$
  $$\gamma_k'^2 = \frac{1}{2}\Big[(a_{xx}-a_{yy})^2+(a_{yy}-a_{zz})^2+(a_{zz}-a_{xx})^2+6(a_{xy}^2+a_{yz}^2+a_{zx}^2)\Big]$$
  where $a_{ij}=\left(\partial\alpha_{ij}/\partial Q_k\right)$.
- Placzek-style activity decomposition:
  $$I_k^{\mathrm{iso}}\propto 45\,\bar{\alpha}_k'^2,\qquad I_k^{\mathrm{ani}}\propto 7\,\gamma_k'^2,\qquad I_k^{\mathrm{tot}}=I_k^{\mathrm{iso}}+I_k^{\mathrm{ani}}$$


In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read
from ase import units
from ase.optimize import BFGS
from mace.calculators.mace import MACECalculator
from mace.calculators import mace_off


/work/mace_alpha_venv/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## Workflow and files used

- Input structure file: `examples/mini_database_IR-R-7193_wB97MD3.xyz`
- Demonstration structure: first frame (`index=0`)
- Model path used in this notebook: `../models/MACE-MDP.model`
- Output folder created by this notebook: `examples/Raman/results`


In [2]:
xyz_path = "../mini_database_IR-R-7193_wB97MD3.xyz"
polar_model_path = "../../models/MACE-MDP.model"

atoms = read(xyz_path, index=0)

## Step 1: Optimize geometry and build normal modes

First, we relax the first XYZ structure with `mace_off(model="medium")` so vibrational analysis starts from a stable geometry.

Then we compute the Hessian, mass-weight it, and diagonalize it to obtain:
- normal mode frequencies in $\text{cm}^{-1}$,
- mass-weighted normal mode displacement vectors used in Raman intensity evaluation.


In [ ]:
device = "cpu"  # switch to "cuda" if available

off_calc = mace_off(model="medium", default_dtype="float64", device=device)
polar_calc = MACECalculator(
    model_paths=polar_model_path,
    model_type="DipolePolarizabilityMACE",
    device=device,
    default_dtype="float64",
)

atoms = atoms.copy()
atoms.calc = off_calc

opt = BFGS(atoms, logfile=None)
opt.run(fmax=0.002, steps=5000)

n_atoms = len(atoms)
masses = atoms.get_masses()

H = off_calc.get_hessian(atoms).reshape(3 * n_atoms, 3 * n_atoms)
H = 0.5 * (H + H.T)

mw = np.repeat(masses ** -0.5, 3)
Hmw = (H * mw).T * mw
omega2, vecs = np.linalg.eigh(Hmw)

conv = units._hbar * units.m / np.sqrt(units._e * units._amu)
energies = conv * np.sqrt(np.abs(omega2))
freq_cm1 = np.real(energies) / units.invcm

modes = vecs.T.reshape(3 * n_atoms, n_atoms, 3)
modes *= masses[np.newaxis, :, np.newaxis] ** -0.5


Using MACE-OFF23 MODEL for MACECalculator with /home/ngoen/.cache/mace/MACE-OFF23_small.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/work/mace_alpha_venv/lib/python3.11/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/work/mace_alpha_venv/lib/python3.11/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


## Step 2: Compute Raman stick intensities from polarizability derivatives

For each normal mode, we displace coordinates by $\pm\Delta Q_k$ and evaluate the polarizability tensor with MACE-MDP.

A central finite difference gives $\partial\boldsymbol{\alpha}/\partial Q_k$, from which we compute:
- isotropic contribution ($45\,\bar{\alpha}_k'^2$),
- anisotropic contribution ($7\,\gamma_k'^2$),
- total Raman activity (sum of isotropic and anisotropic parts).

Modes below 5 $\text{cm}^{-1}$ are set to zero in the example to avoid non-vibrational artifacts.


In [ ]:
total_intensity = []
isotropic_intensity = []
anisotropic_intensity = []

pos0 = atoms.get_positions().copy()
step = 1e-3

for mode_idx, mode in enumerate(modes):
    dpos = mode * step

    atoms.set_positions(pos0 + dpos)
    alpha_plus = polar_calc.get_property("polarizability", atoms)

    atoms.set_positions(pos0 - dpos)
    alpha_minus = polar_calc.get_property("polarizability", atoms)

    atoms.set_positions(pos0)

    dalpha_dq = (alpha_plus - alpha_minus) / (2.0 * step)

    a_iso = np.trace(dalpha_dq) / 3.0
    a_ani_sq = 0.5 * (
        (dalpha_dq[0, 0] - dalpha_dq[1, 1]) ** 2
        + (dalpha_dq[1, 1] - dalpha_dq[2, 2]) ** 2
        + (dalpha_dq[2, 2] - dalpha_dq[0, 0]) ** 2
        + 6.0 * (dalpha_dq[0, 1] ** 2 + dalpha_dq[1, 2] ** 2 + dalpha_dq[2, 0] ** 2)
    )

    if freq_cm1[mode_idx] > 5.0:
        i_iso = 45.0 * (a_iso ** 2)
        i_ani = 7.0 * a_ani_sq
        i_tot = i_iso + i_ani
    else:
        i_iso = 0.0
        i_ani = 0.0
        i_tot = 0.0

    isotropic_intensity.append(i_iso)
    anisotropic_intensity.append(i_ani)
    total_intensity.append(i_tot)

isotropic_intensity = np.array(isotropic_intensity)
anisotropic_intensity = np.array(anisotropic_intensity)
total_intensity = np.array(total_intensity)


## Step 3: Save and inspect the raw stick Raman spectrum

At this stage, we save per-mode frequencies and intensities to CSV. This is the direct numerical output of the finite-difference Raman workflow.

The stick plot helps identify which normal modes dominate the Raman response before any line-shape broadening is applied.


In [ ]:
out_dir = "Raman/results"
os.makedirs(out_dir, exist_ok=True)

raw_path = f"{out_dir}/first_structure_raman_raw.csv"
pd.DataFrame(
    {
        "frequency_cm1": freq_cm1,
        "total_intensity": total_intensity,
        "isotropic_intensity": isotropic_intensity,
        "anisotropic_intensity": anisotropic_intensity,
    }
).to_csv(raw_path, index=False)

plt.figure(figsize=(8, 4.5))
plt.vlines(freq_cm1, 0.0, total_intensity, lw=1.0, label="Total")
plt.xlim(0, 4000)
plt.xlabel(r"Raman shift ($\text{cm}^{-1}$)")
plt.ylabel("Stick intensity (arb. units)")
plt.title("Raman stick spectrum (first XYZ structure)")
plt.tight_layout()
plt.show()

print(f"Wrote: {raw_path}")


## Step 4: Apply Lorentzian broadening for visualization

We convert the stick spectrum into a smooth curve using Lorentzian broadening. This is a post-processing step for plotting and comparison; it does not change the underlying normal-mode frequencies.

The notebook writes the broadened total, isotropic, and anisotropic spectra to CSV.


In [ ]:
def broaden_spectrum_lorentzian(freqs, intens, fwhm=10.0, x_min=0.0, x_max=4000.0, step=1.0):
    x = np.arange(x_min, x_max + step, step, dtype=np.float64)
    gamma = 0.5 * fwhm
    spec = np.zeros_like(x)
    for f0, I in zip(freqs, intens):
        spec += I * (gamma / ((x - f0) ** 2 + gamma ** 2))
    return x, spec

x_cm1, spec_total = broaden_spectrum_lorentzian(freq_cm1, total_intensity)
_, spec_iso = broaden_spectrum_lorentzian(freq_cm1, isotropic_intensity)
_, spec_ani = broaden_spectrum_lorentzian(freq_cm1, anisotropic_intensity)

norm = spec_total.max() if spec_total.max() > 0 else 1.0
spec_total /= norm
spec_iso /= norm
spec_ani /= norm

broad_path = f"{out_dir}/first_structure_raman_broadened.csv"
pd.DataFrame(
    {
        "cm1": x_cm1,
        "total": spec_total,
        "isotropic": spec_iso,
        "anisotropic": spec_ani,
    }
).to_csv(broad_path, index=False)

plt.figure(figsize=(8, 4.5))
plt.plot(x_cm1, spec_total, label="Total", lw=1.5)
plt.plot(x_cm1, spec_iso, label="Isotropic", alpha=0.8)
plt.plot(x_cm1, spec_ani, label="Anisotropic", alpha=0.8)
plt.xlim(0, 4000)
plt.xlabel(r"Raman shift ($\text{cm}^{-1}$)")
plt.ylabel("Normalized intensity")
plt.title("Raman broadened spectrum (first XYZ structure)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Wrote: {broad_path}")


## Interpreting results

- `first_structure_raman_raw.csv` contains per-mode Raman activities.
- `first_structure_raman_broadened.csv` contains smooth curves for comparison/plotting.
- Strong peaks correspond to normal modes with large polarizability derivatives.
